In [10]:
import pandas as pd
import ollama
from tqdm import tqdm
tqdm.pandas()


# Load data
df = pd.read_csv("injuries_scraped_fast_safe.csv")

# Remove rows where Notes or Relinquished is missing
df = df.dropna(subset=['Notes', 'Relinquished'])

# Filter out activation/IL records
df = df[~df['Notes'].str.lower().str.contains("activated from il")]

# Function to extract injury location
def extract_injury_location(note):
    try:
        response = ollama.chat(
            model="llama3",
            messages=[{
                "role": "user",
                "content": f"Extract just the body part injured in this note: '{note}'. Respond with only the body part, one or two words, and nothing else. Side of the body doesn't matter so if it's left ankle it's just ankle."
            }]
        )
        return response['message']['content'].strip()
    except Exception as e:
        return f"Error extracting injury location: {e}"

# Apply LLM extraction 
df['Injury Location'] = df['Notes'].progress_apply(extract_injury_location)

# Replace Notes with extracted result
df['Notes'] = df['Injury Location']
df.drop(columns=['Injury Location'], inplace=True)

# Save result
output_file = "modified_injury_locations_scraped.csv"
df.to_csv(output_file, index=False)
print(f"✅ Data saved to {output_file} ({len(df)} rows)")


100%|██████████| 5655/5655 [26:57<00:00,  3.50it/s]  

✅ Data saved to modified_injury_locations_scraped.csv (5655 rows)


Remove rows with no injury etc

In [14]:
import pandas as pd

# Load the processed dataset
df = pd.read_csv("modified_injury_locations_scraped.csv")
original_count = len(df)

# Standardize notes: convert to string, strip spaces
df = df.dropna(subset=['Notes'])
df['Notes'] = df['Notes'].astype(str).str.strip()

# Track how many rows removed
removed_total = 0

# Step 1: Remove rows with 'rest' 'dnp' 'none'
before = len(df)
df = df[~df['Notes'].str.lower().isin(['rest', 'dnp', 'none'])]
after = len(df)
print(f"Removed {before - after} rows with 'rest', 'none', or 'dnp'")
removed_total += before - after

# Step 2: Remove rows that start with 'Error'
before = len(df)
df = df[~df['Notes'].str.startswith("Error")]
after = len(df)
print(f"Removed {before - after} rows with LLM errors")
removed_total += before - after

# Step 3: Remove rows with more than 2 words
before = len(df)
df = df[df['Notes'].str.split().str.len() <= 2]
after = len(df)
print(f"Removed {before - after} rows with long descriptions")
removed_total += before - after

# Reset the index for a clean DataFrame
df = df.reset_index(drop=True)

# Save the cleaned dataset
df.to_csv("injury_locations_scraped_cleaned.csv", index=False)

# Final summary
print(f"\nOriginal rows: {original_count}")
print(f"Final rows: {len(df)}")
print(f"Total removed: {removed_total}")


Removed 0 rows with 'rest', 'none', or 'dnp'
Removed 0 rows with LLM errors
Removed 138 rows with long descriptions

Original rows: 5655
Final rows: 4633
Total removed: 138


In [24]:
import pandas as pd

df = pd.read_csv("injury_locations_categorized.csv")

# Clean the Notes column to avoid type issues
df['Injury Category'] = df['Injury Category'].astype(str).str.strip()

# Drop rows that are empty or 'nan'
df = df[df['Injury Category'].str.lower() != 'nan']

# Get sorted unique injuries
unique_injuries = sorted(df['Injury Category'].unique())
print(f"Found {len(unique_injuries)} unique injury locations:\n")
for injury in unique_injuries:
    print(injury)


Found 24 unique injury locations:

abdomen
ankle
arm
back
chest
eye
face
finger
foot
glute
groin
hand
hip
internal
knee
leg
lungs
neck
seasonal/other
shoulder
teeth
throat
toe
wrist


In [23]:
df = pd.read_csv("injury_locations_scraped_cleaned.csv")
df['Notes'] = df['Notes'].astype(str).str.strip().str.lower()    .str.replace(r'\s+', ' ', regex=True)


injury_mapping = {
    "ankle": [
        "ankle", "foot ankle", "foot/ankle", "leg ankle", "tibia ankle",
        "soreness/ankle", "ankle ", "ankle.", "ankle/knee", "heel", "ankle elbow",
        "ankle shoulder", "back ankle", "ankle wrist"
    ],
    "knee": [
        "knee", "knees", "knee cap", "knee meniscus", "kneecap", "knee tendon",
        "knee tendons", "knee, ankle", "knee hip", "knee ankle", "meniscus knee",
        "patella", "meniscus", "calf knee", "ankle knee", "hip knee", "knee hamstring",
        "knee shoulder", "hnee", "knee back", "knee, foot", "right knee"
    ],
    "foot": [
        "foot", "foot.", "right foot", "arch", "metatarsal", "midfoot", "foot/toe"
    ],
    "toe": ["toe", "big toe", "little toe", "toe thumb", "toenail", "thumb toe", "to"],
    "finger": [
        "finger", "index finger", "middle finger", "ring finger", "pinky finger",
        "little finger", "hand/finger", "right hand/finger", "thumb", "thumb ankle"
    ],
    "hip": ["hip", "hip flexor", "hip pointer", "hip/knee", "pelvis", "tailbone", "symphysis", "si joint", "hip knee"],
    "shoulder": ["shoulder", "neck shoulder", "rotator cuff", "axilla", "collarbone", "labrum"],
    "back": ["back", "lower back", "upper back", "spinal cord", "lat", "back calf", "back knee",
             "lumbar", "lumbar spine", "spine"
            ],
    "wrist": ["wrist", "hand/wrist", "wrist ankle", "wrist knee"],
    "hand": ["hand", "right hand", "hand/thumb"],
    "chest": ["chest", "chest muscle", "pectoral", "pectoral muscle", "pectoralis", "thorax", "sternum", 
              "stemum", "chest shoulder", "ribs knee", "rib", "ribs", "ribcage"
    ],
    "abdomen": ["abdomen", "abdominal muscle", "gut", "stomach", "hernia", "oblique", "core", "abdomen hip", "side"],
    "groin": ["groin", "pubic area", "adductor", "adductor muscle", "abductor", "testicle", "groin hip"],
    "leg": [
        "leg", "right leg", "left leg", "calf", "calf/shin", "shin", "tibia",
        "achilles", "fibula", "thigh", "quad", "quadriceps", "quadricep", "quadricap", "hamstring", "hamstrings", 
        "leg ankle", "meniscus calf", "ankle calf", "calf elbow", "foot hamstring", "foot knee", "hamstring back", 
        "hamstring knee", "leg tibia", "calf ankle", "hamstring hip", "hamstring rib", "lower leg", "quadriceps back"
    ],
    "arm": ["arm", "bicep", "biceps", "forearm", "tricep", "triceps", "elbow"],
    "face": ["face", "cheek", "cheekbone", "jaw", "mouth", "nose", "facial bone", "head", "brain", "ear", "forehead", "facial"],
    "neck": ["neck", "cervical"],
    "eye": ["eye", "cornea", "orbital", "orbital wall", "orbital bone", "orbital floor", "eyelid"],
    "lungs": ["lung", "lungs", "respiratory", "respiratory system"],
    "throat": ["throat", "tonsils", "sinus", "sinuses"],
    "teeth": ["tooth", "teeth", "wisdom tooth", "dental"],
    "glute": ["glute", "gluteus", "glutes"],
    "muscle": ["fascia", "muscle"],
    "internal": ["intestine", "intestines", "append", "appendix", "heart", "heath", "gastrointestinal"],
    "seasonal/other": ["season", "health", "nothing", "no injury", "root", "dtd", "blood", "skin", "body",
                       "other", "illness", "Unknown", "Undisclosed", "various", "various injuries"]
}


flat_map = {term.lower(): category for category, terms in injury_mapping.items() for term in terms}

def normalize(note):
    note_clean = note.lower().strip()
    return flat_map.get(note_clean, "misc")

df['Injury Category'] = df['Notes'].apply(normalize)

uncategorized = df[df['Injury Category'] == "misc"]['Notes'].unique()

print(f"\n⚠️ Found {len(uncategorized)} uncategorized injury terms:\n")
for term in sorted(uncategorized):
    print(term)

df.to_csv("injury_locations_categorized.csv", index=False)
print(f"\nFinal categorized dataset saved to injury_locations_categorized.csv with {df['Injury Category'].nunique()} categories.")



⚠️ Found 0 uncategorized injury terms:


Final categorized dataset saved to injury_locations_categorized.csv with 24 categories.


In [3]:
import pandas as pd
import re

def normalize_name(name):
    if pd.isna(name):
        return ""

    # Remove leading bullet points and extra spaces
    name = name.replace("•", "").strip()
    
    # If multiple players, split on " / " and keep the first player
    if "/" in name:
        name = name.split("/")[0].strip()

    # If name has parentheses like "Justin Jackson (Aaron)", keep only before "("
    if "(" in name:
        name = name.split("(")[0].strip()

    # Remove extra spaces again
    name = " ".join(name.split())

    # Title-case the name (first letters capitalized)
    name = name.title()

    return name

df = pd.read_csv("injury_locations_scraped_categorized.csv")
df['Relinquished'] = df['Relinquished'].apply(normalize_name)
df.to_csv("injury_locations_scraped_categorized_named.csv", index=False)

# Join the kaggle set and scraped set to create a data set over the last 15 years

In [9]:
import pandas as pd
import re

# Step 1: Load your cleaned + categorized injury dataset
categorized_df = pd.read_csv("/Users/tymandachit/Documents/SW/MachineLearningProject/Cleaned Data/Finished/injury_locations_categorized.csv")
scraped_df = pd.read_csv("/Users/tymandachit/Documents/SW/MachineLearningProject/Cleaned Data/Finished/injury_locations_scraped_categorized_named.csv")

print(f"categorized_df size: {len(categorized_df)}")
print(f"scraped_df size: {len(scraped_df)}")

# Step 2: Drop nulls and Acquired
categorized_df = categorized_df.dropna(subset=['Relinquished'])

if 'Acquired' in categorized_df.columns:
    categorized_df = categorized_df.drop(columns=['Acquired'])

scraped_df = scraped_df.dropna(subset=['Relinquished'])

if 'Acquired' in scraped_df.columns:
    scraped_df = scraped_df.drop(columns=['Acquired'])

# Step 3: Join (combine) them together
merged = pd.concat([categorized_df, scraped_df], ignore_index=True)

# Save the final merged file
merged = merged.drop(columns=['Notes'])
merged.to_csv("player_injures_2010-2025.csv", index=False)
print(f"✅ Merged dataset saved with {len(merged)} rows to injury_locations_combined.csv")

categorized_df size: 9099
scraped_df size: 4607
✅ Merged dataset saved with 13556 rows to injury_locations_combined.csv
